# 02 — Data Cleaning, Preprocessing, and Leakage-Free Splitting

**Project:** Obstetric Ultrasound Segmentation and Fetal Head Circumference (HC) Measurement (SBES426 — Team 2)  
**Goals:**
1. Audit and resolve the **3 exact duplicate file groups** discovered in Phase 1 EDA.
2. Flood-fill hollow `*_Annotation.png` ellipse contours into solid binary masks (`1` = fetal head, `0` = background).
3. Standardize image/mask dimensions (accounting for the 48 non-800×540 images) and rescale effective `pixel size (mm/px)` along x/y axes.
4. Normalize intensities and apply **edge-preserving speckle reduction** (saving both normalized-only and despeckled variants for ablation).
5. Create a leakage-free **70% / 15% / 15% Train / Validation / Internal Test** split grouped by exam numeric prefix ID (`000`–`805`) and stratified across HC trimester/size bins.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    RAW_TRAIN_DIR,
    RAW_TRAIN_CSV,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    SPLITS_DIR,
    FIGURES_DIR,
    COL_FILENAME,
    COL_PIXEL_SIZE,
    COL_HC_MM,
    RANDOM_SEED,
    DEFAULT_TARGET_SIZE,
)

print("Raw train dir:", RAW_TRAIN_DIR)
print("Raw train CSV:", RAW_TRAIN_CSV)

## 1. Duplicate Audit & Resolution
- Hash all ultrasound `.png` files in `data/raw/training_set/` (ignoring `Thumbs.db`).
- Inspect the 3 duplicate groups alongside their CSV `pixel size`, `head circumference (mm)`, and `*_Annotation.png` files.
- Save the cleaned metadata manifest to `data/interim/cleaned_metadata.csv`.

In [ ]:
# TODO: Implement duplicate detection and cleaning via src/data_prep/

## 2. Flood-Fill Hollow Ellipse Annotations into Solid Binary Masks
- Convert each `*_Annotation.png` contour into a filled binary mask (`uint8`, values `{0, 1}` or `{0, 255}`).
- Verify mask area and foreground/background pixel ratio (quantifying pixel-level class imbalance).
- Visually inspect random filled masks overlaid on ultrasound images.

In [ ]:
# TODO: Implement contour flood-fill and save solid masks to data/interim/masks/

## 3. Resizing, Pixel-Size Recalibration, Normalization & Edge-Preserving Despeckling
- Standardize images and binary masks to `DEFAULT_TARGET_SIZE` (`H x W`), recording `pixel_size_x_mm` and `pixel_size_y_mm`.
- Normalize image intensities to `[0, 1]`.
- Apply edge-preserving speckle-reduction filtering (e.g., Bilateral, Anisotropic Diffusion / Perona-Malik, SRAD, or Non-Local Means) and plot before/after comparisons.

In [ ]:
# TODO: Run preprocessing pipeline from src/preprocessing/ and export to data/processed/

## 4. Exam-Grouped, HC-Stratified Train / Validation / Test Split (70% / 15% / 15%)
- Extract numeric exam prefix ID from `filename` (e.g., `10_HC.png` and `10_2HC.png` -> exam `10`).
- Bin HC into trimester/size strata so 1st, 2nd, and 3rd trimester scans are proportionally represented across splits.
- Assert **zero exam ID overlap** across Train, Val, and Test.
- Save `train.csv`, `val.csv`, and `test.csv` to `data/splits/`.

In [ ]:
# TODO: Generate and validate splits via src/data_prep/ and save to SPLITS_DIR